# Dependencies

In [1]:
import sys
sys.path.insert(0, "../input/chenglu-ai4code-source")

In [2]:
try:
    from nltk.stem import WordNetLemmatizer
except:
    !pip install ../input/nltk37/nltk/nltk-3.7-py3-none-any.whl
    !cp -r ../input/nltk37/nltk/nltk_data ~/
    from nltk.stem import WordNetLemmatizer
import pickle
import ai4code
import os
import multiprocessing

ModuleNotFoundError: No module named 'ai4code'

In [3]:
METRIC_SPLIT_MODE = "max"
ENSEMBLE_MODE = "weighted_average"

In [4]:
try:
    import featurize
    IN_FEATURIZE = True
    CODEBERT_PRETRAINED_PATH = "/home/featurize/codebert-base"
    DEBERTA_PRETRAIN_PATH = "/home/featurize/deberta-v3-small/"
    DEBERTA_BASE_PRETRAIN_PATH = "../input/deberta-v3/deberta-v3-base"
except:
    IN_FEATURIZE = False
    CODEBERT_PRETRAINED_PATH = "../input/codebert/codebert-base"
    DEBERTA_PRETRAIN_PATH = "../input/deberta-v3/deberta-v3-small"
    DEBERTA_BASE_PRETRAIN_PATH = "../input/deberta-v3/deberta-v3-base"

# Configuration

In [5]:
ai4code.cfg = cfg = ai4code.utils.Config(
    dataset_root = "../input/AI4Code" if not IN_FEATURIZE else "/home/featurize/data/",
    encode_files = [
        {
            "name": "deberta-v3-v8",
            "preprocessor": "preprocessor_v8",
            "pretrain_path": DEBERTA_PRETRAIN_PATH,
        },
#         {
#             "name": "codebert-v8",
#             "preprocessor": "preprocessor_v8",
#             "pretrain_path": CODEBERT_PRETRAINED_PATH,
#         }
    ],
    checkpoints = [
#         {
#             "path": "../input/kafka-deberta-base/checkpoint_570000_kendall_tau0.9108.pt",
#             "encode": "deberta-v3-v8",
#             "pretrain_path": DEBERTA_BASE_PRETRAIN_PATH,
#             "most_common_encodes": "../input/debertav3-most-common-encodes/most_common_encodes.pkl",
#         },

        {
            "path": "../input/kafka-deberta-small-2/checkpoint_56102_kendall_tau0.9378.pt",
            "encode": "deberta-v3-v8",
            "pretrain_path": DEBERTA_PRETRAIN_PATH,
            "most_common_encodes": "../input/debertav3-most-common-encodes/most_common_encodes.pkl",
        },
#         {
#             "path": "../input/dancing-mad-code-bert-1/checkpoint_200000_kendall_tau0.9343.pt",
#             "encode": "codebert-v8",
#             "pretrain_path": CODEBERT_PRETRAINED_PATH,
#             "most_common_encodes": None
#         }
    ],
)


NameError: name 'ai4code' is not defined

In [6]:
ai4code.utils.dump_cfg(cfg)

NameError: name 'ai4code' is not defined

# Tokenization

Tokenize all the text in a standalone process to ensure memory been fully released after.

In [7]:
%%file /tmp/gen.py

import pickle

try:
    import featurize
    IN_FEATURIZE = True
    val_sample_ids = list(pickle.load(open("/home/featurize/work/ai4code/data/v8_deberta_small/0.pkl", "rb")).keys())
    print("val sample num: ", len(val_sample_ids))
except Exception as e:
    IN_FEATURIZE = False

import sys
sys.path.insert(0, "../input/chenglu-ai4code-source")
from transformers import AutoTokenizer
import ai4code
import multiprocessing
from tqdm import tqdm
from pathlib import Path
import pandas as pd
from functools import reduce

ai4code.cfg = ai4code.utils.load_cfg()
dataset_root = Path(ai4code.cfg.dataset_root)

orders = pd.read_csv(dataset_root / "train_orders.csv")
ai4code.utils.orders_dict = {}
for _, item in orders.iterrows():
    ai4code.utils.orders_dict[item.id] = item.cell_order.split(" ")

for encode_file in ai4code.cfg.encode_files:
    encode_file["tokenizer"] = AutoTokenizer.from_pretrained(encode_file["pretrain_path"], do_lower_case=True, use_fast=True)
    encode_file["preprocessor"] = getattr(ai4code.datasets.preprocessor, encode_file["preprocessor"])
    # dump special tokens at first
    pickle.dump(dict(
        hash_id=encode_file["tokenizer"].encode("#", add_special_tokens=False)[0],
        cls_token_id=encode_file["tokenizer"].cls_token_id,
        sep_token_id=encode_file["tokenizer"].sep_token_id,
        pad_token_id=encode_file["tokenizer"].pad_token_id,
        unk_token_id=encode_file["tokenizer"].unk_token_id,
    ), open(f"./special_tokens.{encode_file['name']}.pkl", "wb"))

with multiprocessing.Pool(processes=multiprocessing.cpu_count()) as pool:
    if IN_FEATURIZE:
        json_files = list((dataset_root / "train").glob("*.json"))
        json_files = [json_file for json_file in json_files if json_file.name.split(".")[0] in val_sample_ids][:500]
    else:
        json_files = list((dataset_root / "test").glob("*.json"))
        # json_files = list((dataset_root / "train").glob("*.json"))[:100]
    print("json files total:", len(json_files))

    results = list(
        tqdm(
            pool.imap(ai4code.utils.process, json_files),
            total=len(json_files),
        )
    )

all_data = {sample.id: sample for sample in results}
pickle.dump(all_data, open("/tmp/encodes.pkl", "wb"))

Writing /tmp/gen.py


In [8]:
if not os.path.exists("/tmp/encodes.pkl"):
    !python /tmp/gen.py

NameError: name 'os' is not defined

# Inference

In [9]:
import torch
import transformers
from transformers import AutoTokenizer
import pandas as pd
import multiprocessing
from tqdm import tqdm
from pathlib import Path
import os
import pickle

In [10]:
enable_amp = True if IN_FEATURIZE else False
batch_size = 128 if IN_FEATURIZE else 128
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [11]:
print("torch version:", torch.__version__)
print("transformers version:", transformers.__version__)

torch version: 2.6.0+cu124
transformers version: 4.53.3


In [12]:
with open("/tmp/encodes.pkl", "rb") as f:
    data = pickle.load(f)

FileNotFoundError: [Errno 2] No such file or directory: '/tmp/encodes.pkl'

In [13]:
# TODO: 提前创建 dataset ，创建完后删除 encode 节约内存
def create_dataset(checkpoint):
    state = torch.load(checkpoint['path'])
    params = state["params"]
    samples_dir = Path("/tmp/samples") / checkpoint['encode']
    samples_dir.mkdir(exist_ok=True, parents=True)
    del state
    model_encode = checkpoint['encode']
    special_tokens = ai4code.datasets.SpecialTokenID(
        **pickle.load(open(f"./special_tokens.{model_encode}.pkl", "rb"))
    )

    dataset = ai4code.datasets.MixedDatasetWithSplits(
        data,
        special_tokens,
        only_task_data=True,
        encode_key=model_encode,
        anchor_size=params["val_anchor_size"] if "val_anchor_size" in params else params["anchor_size"],
        max_len=params["max_len"],
        split_len=params["split_len"],
        global_keywords=checkpoint["most_common_encodes"]
    )
    dataset.presist_samples(samples_dir)
    checkpoint["_dataset"] = dataset

    # remove all the encode
    for sample in tqdm(data.values()):
        sample.cell_encodes[checkpoint['encode']] = None


for checkpoint in ai4code.cfg.checkpoints:
    create_dataset(checkpoint)

NameError: name 'ai4code' is not defined

In [14]:
model_args = ("with_lstm", "with_context_feature", "max_len")
submission_data = []

def inference(checkpoint, k):
    model_encode = checkpoint['encode']
    state = torch.load(checkpoint['path'])
    params = {
        **state["params"],
        **({} if "params" not in checkpoint else checkpoint["params"]),
    }
    ai4code.utils.print_params(params)
    model_params = {k: v for k, v in params.items() if k in model_args}
    model = ai4code.models.MultiHeadModel(checkpoint['pretrain_path'], with_lm=False, dropout=0, **model_params)
    if "n_averaged" in state['model']:
        state['model'] = {k.replace("module.", ""): v for k, v in state['model'].items()}
    try:
        model.load_state_dict(state['model'])
    except Exception as e:
        print(f"load model failed with strict mode: {e}, try to load with unstrict")
        model.load_state_dict(state['model'], strict=False)
    model.eval()
    model.to(DEVICE)
    del state

    loader = torch.utils.data.DataLoader(
        checkpoint["_dataset"],
        num_workers=2,
        batch_size=batch_size,
    )

    metric = ai4code.metrics.KendallTauWithSplits(data, params["split_len"], mode=METRIC_SPLIT_MODE)

    with torch.no_grad():
        for batch in tqdm(loader):
            ids, mask, targets = [item.to(DEVICE) for item in batch[:3]]
            sample_ids, cell_keys, split_ids, rank_offset = batch[3:]
            with torch.cuda.amp.autocast(enabled=enable_amp):
                in_split, rank, _ = model(ids, mask, lm=False)
            metric.update((0, in_split, rank, sample_ids, cell_keys, split_ids, rank_offset))
        print("Score: ", metric.compute())
        with open(f"/tmp/raw_preds.{k}.pkl", "wb") as f:
            pickle.dump(metric._raw_preds, f)

for k, checkpoint in enumerate(ai4code.cfg.checkpoints):
    inference(checkpoint, k)

NameError: name 'ai4code' is not defined

In [15]:
all_raw_preds = [pickle.load(open(f"/tmp/raw_preds.{k}.pkl", "rb")) for k, _ in enumerate(ai4code.cfg.checkpoints)]

keys = all_raw_preds[0].keys()
all_ensemble_preds = {}

for sample_id, sample in tqdm(data.items()):
    sample_raw_preds = [raw_preds[sample_id] for raw_preds in all_raw_preds]
    sample_ensemble_pred = []
    ensembled_cell_preds = []
    for cell_preds in zip(*sample_raw_preds):
        if ENSEMBLE_MODE == "weighted_average":
            cell_pred = list(cell_preds[0])
            if sample.cell_types[cell_pred[0]] == "markdown":
                scores = torch.sigmoid(torch.tensor([p[2] for p in cell_preds]))
                ranks = torch.tensor([p[1] for p in cell_preds])
                weights = scores / scores.sum()
                rank = (ranks * weights).sum()
                cell_pred[1] = rank
            ensembled_cell_preds.append(cell_pred)
        else:
            cell_pred = cell_preds[0]
            if sample.cell_types[cell_pred[0]] == "markdown":
                for cur_cell_pred in cell_preds[1:]:
                    if cur_cell_pred[2] > cell_pred[2]:
                        cell_pred = cur_cell_pred
            ensembled_cell_preds.append(cell_pred)
    cell_id_predicted = [
        item[0] for item in sorted(ensembled_cell_preds, key=lambda x: x[1])
    ]
    all_ensemble_preds[sample_id] = cell_id_predicted

NameError: name 'ai4code' is not defined

In [16]:
if os.path.exists("../input/AI4Code/sample_submission.csv"):
    sample_df = pd.read_csv("../input/AI4Code/sample_submission.csv")
    for idx, sample_id in enumerate(sample_df.id):
        sample_df.loc[idx, 'cell_order'] = " ".join(all_ensemble_preds[sample_id])
    sample_df.to_csv("submission.csv", index=False)

NameError: name 'all_ensemble_preds' is not defined